# Team Member Contribution Table

| Member      | Contribution |
| ----------- | ----------- |
| Petter      | Managed software parts, including adding new API enpoints and dependency managements.|
| Kim         | Worked on classification models. |
| Elliot      | Researched on what accuracy metric to use. Looked into how we can integrate MLflow. |
| Agust       | Managed software parts, including extending the frontend to allow the user to request predictions from the new model. |

## Machine Learning Process Iterations

This section documents the machine learning-specific development process, with emphasis on iterative model improvement and dependency management, as required by the project guidlines. All descriptions below are limited to the machine learning responsibility of the project.

## 1. Iterations over the machine learning process
The machine learning pipeline was not implemented in a single step, but iteratively refined by revisiting several core ML process stages. Each iteration was motivated by empirical findings from evlatuiation metrics and diagnostic visualizations.

### 1.1 Initial problem formulation and baseline

The task was formulated as a multi-class classification problem, where the goal was to predict the weakest link in a movement based on 38 numerical predictors derived from angle deviation and NASM deviation measurements. Each sample belonged to exactly one weakest link class, and all classes were initially assumed to be equally valid targets.

As a starting point, a baseline model was implemented using a **DummyClassifier** that always predicts the most frequent class. Thisd model requires no learning and serves as a lower bound for performance. The baseline was evaluated using accuracy as well as macro-averaged metrics, particularly **macro F1 score**, which treats all classes equally regardless of size.

The baseline achived a very low macro F1 score, despite a non-trivial accuracy value. This immediatwely revelade two key issues:
* Strong imbalance 
* Inadequacy of accuracy as a sole metric
This initial iteration validated the choice of macro-averaged metrics and established a reference point against which all subsequent models were compared.

### 1.2 First model comparison iteration

After establishing the baseline, the next iteration focused on comparing a broad set of fundamentally different model families. Before training, feature scaling was introduced using **StandardScaler**, which transforms each feature to have approximately zero mean and unit standard deviation. This step was necessay for models that rely on distance calculations or assume comparable feature scales, such as Logistic Regression, k-Nearest Neighbors, and Linear Discriminant Analysis.

Several classical machine learning models were evaluated under identical conditions, including Logistic Regression, Gaussian Naive Bayes, k-Nearest Neighbors, and Linear Discriminant Analysis. All models were trained and evaluated using stratified cross-validation, ensuring that each fold preserved the original class distribution. 

Evaluation was performed using accuracy, macro-averaged precision, recall, macro F1 score, and weighted F1 score. The results showed that linear models substantially outperformed the baseline, while accuracy alone remained misleading due to class imbalance. Macro F1 score emerged as the primary criterion for model comparison, as it better reflected performance across minority classes.

This iteration amphgazised breadth over depth and established a reliable ranking of model families under consistent preprocessing and evaluation conditions.

### 1.3 Label quality and class distribution iteration

Analysis of per-class performance revealed significant issues realted to label quality and class distribution. Some weakest link classes contined extremely few samples, in some cases fewer than two. Additionally, certain samples exhibited *ties*, meaning multiple indicators shared the maximum deviation value, resultiong in ambigiouos weakest link labes. 

These issues manifested as near-zero per-class F1 scores, unstable stratified cross-validation splits, and high variance in evaluation metrics. To address this, a **HasTie** flag was introduced to explicitly identify ambiguous samples, and weakest link classes with fewer than two samples were removed from the dataset. Further empirical analysis motivated the removal of additional extremely rare classes tha consistently degraded model stability.

This iteration did not change the model architecture but instead focused on **data quality**, improving label reliability and reducing noise in the trainingh set. As as result, cross-validation became more stable and model performance more interpretable.

### 1.4 Cross-validation strategy iteration
Initially, the minimum class size constrained stratified cross-validation to only two folds, limiting the statistical reliability of performance estimates. After removing rare classes, the smallest class size increased to allow five-fold strativied cross-validation.

Increasing the number of folds reduced variance in evaluation metrics and improved confidence in model comparisons. This iteration refined the evaluation procedure itself, ensuring that observed performance differences were meaningful rather than artifacts of unstable data splits.

### 1.5 Hyperparameter sensitivity iteration

With a cleaner dataset and amore reliable evaluation strategy, attention shifted to hyperparmeter sensitvity, particularly the reqularization strenght in Logistic Regression. The regularization paremeter **C** controls the trade-off between model complexity and generalization. Lower values enforce stronger regularization, while higher values allow the model to fit the data more closely

Empirical results showed that after removing noisy and rare classes, models with higher **C** values (weaker regularization) consistently outperformed more strongly regularized variants. This behaviour indicates that the cleaned dataset contained a stronger and more coherent signal, reducing the risk of overfitting and allowing the model to benefit from increased flexibility.

This iteration demonstrated the interdependence between data preprocessing and model tuning, highlighting that optimal hyperparameters depend stronglynon dataset quality.

### 1.6 Advanced model iteration: Boosting models
In the final iteration, more expressive models were introduced, including XGBoost and LightGBM. These gradient boosting models were selected for their ability to capture non-linear relationships and complex feature interactions that lienar models cannot represent.

Under the same cross-validation and preprocessig setup, boosting models achived the highest macro F1 scores across all experiments. XGBoost consistently ranked as the strongest overall model, demonstrating improved balance between majority and minority class performance.

This iteration marked the final expansion of model complexity and was justified by tghe progressively improved data quality and evaluation robustness established in earlier steps.

# Performance Metrics Report: Weakest Link Classification

## 1. Metrics Used

### Primary Metrics

| Metric | Purpose |
|--------|---------|
| **F1-macro**  | Primary metric, treats all classes equally |
| F1-weighted | Weighted by class frequency |
| Accuracy mean| Overall correctness |
| Precision (macro) | Average precision across classes |
| Recall (macro) | Average recall across classes |

### Visual Metrics

- **Confusion Matrix**: Shows actual misclassification patterns


## 2. Why These Metrics?

Our dataset has extreme imbalance:

```
Class Distribution (Test Set):
- LeftArmFallForward:      126 samples (30%)
- RightArmFallForward:      94 samples (22%)
- RightKneeMovesOutward:    55 samples (13%)
- RightShoulderElevation:   50 samples (12%)
- ...
- LeftKneeMovesInward:      1 sample  (<1%)
- RightHeelRises:           0 samples
```

### Why F1-Macro is Primary

**Macro-averaging treats all classes equally**, regardless of frequency.

1. Rare movemements are as important as common ones
   - A model that only detects common issues is clinically inadequate
   - Rare movement deviation patterns may indicate serious problems

2. Prevents Gaming the Metric: 
   - Simple accuracy can be misleading (baseline "predict most frequent" = 30% accuracy)
   - Weighted-F1 rewards performance on common classes only
   - Macro-F1 forces the model to learn all classes for a good score

3. **Balances Precision and Recall**:
   - F1 is the harmonic mean of precision and recall
   - Ensures the model both finds deficiencies (recall) and correctly identifies them (precision)


## 3. Complementary Metrics

### F1-Weighted
- **Purpose**: Shows performance weighted by class frequency
- **Insight**: Higher than macro-F1, indicating better performance on common classes
- **Interpretation**: Model is decent at frequent patterns but struggles with rare ones

### Accuracy
- **Purpose**: Overall correctness percentage
- **Limitation**: Misleading for imbalanced data such as ours
- **Insight**: Better than baseline (30%), but doesn't show per-class performance

### Per class F1
![Confusion matrix](img/PerClassF1.png)

- **Purpose**: How the model performs for each class
- **Interpretation**: Again, the models struggles with rare classes


### Confusion Matrix
![Confusion matrix](img/confusion_matrix.png)

- **Purpose**: Understand misclassification patterns
- **What we found**:
  - Left/Right confusion
  - Sometimes arms confused with knees

# Dependency management

To ensure reliable and reproducible dependency management, we use python venv and npm for the FastAPI backend and the web application frontend, respectively.

For the backend, Python venv is used to create an isolated virtual environment that contains all required dependencies for the FastAPI server, keeping them separate from system-wide packages. All backend dependencies are specified in requirements.txt. Setup instructions are provided in ml/api/README.txt. 

For the frontend, npm is used to install and manage all necessary JavaScript dependencies, as defined in package.json and locked via package-lock.json to ensure consistent installations across environments. Setup instructions are available in sw/README.txt

Instead of manually configuring and running each application independently, we use Docker and Docker compose to containerize both applications and orchestrate them as a single system. This approach simplifies setup, and enables seamless service-to-service and user-to-service communciation. Setup instructions are available in README.txt.

For summary, dependency management is enforced through: 
* Use of provided setup instructions
* Version locking in dependency files
* Docker-based builds to prevent the use of undeclared or system-wide dependencies




# Comments

* For the next iteration we will be starting to use mlflow for training and tracking models.